# AT1 - Perceptron e KNN em Prática

Aprendizagem de Máquina - Fatec

Aluno: Yago Uran Kurashiki Rios

Esse notebook tem os três desafios da atividade:

1. Perceptron com treinamento automático (fraude em transações)
2. Classificador KNN (risco de churn)
3. Recomendador por similaridade (servidores cloud)

In [ ]:
import numpy as np

## Desafio 1 - Perceptron (detecção de fraude)

Cada transação tem duas características:

- x1: valor da transação normalizado (0 a 10)
- x2: quantidade de operações nos últimos 5 minutos

A saída é 0 (legítima) ou 1 (suspeita).

In [ ]:
X_treino_fraude = np.array([
    [1.5, 1.0], [2.0, 2.0], [3.5, 1.5], [3.0, 3.0],
    [6.5, 5.0], [7.0, 7.0], [8.5, 6.0], [9.0, 8.0]
])
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

### Função de inferência

Faz a combinação linear `z = w . x + b` e passa na função degrau: se z >= 0 a classe é 1, senão é 0.

Deixei ela antes do treino porque o treino usa essa mesma função pra prever.

In [ ]:
def prever_perceptron(x, w, b):
    z = np.dot(x, w) + b
    if z >= 0:
        return 1
    return 0

### Função de treinamento

Pesos e viés começam em zero. Em cada época passo por todas as amostras e aplico a regra de Rosenblatt:

- erro = y real - y previsto
- w = w + eta * erro * x
- b = b + eta * erro

Quando o modelo acerta o erro é 0 e nada muda. Também conto os erros de cada época só pra acompanhar se está aprendendo.

In [ ]:
def treinar_perceptron(X, y, eta, epocas):
    w = np.zeros(X.shape[1])
    b = 0.0

    for epoca in range(epocas):
        erros = 0
        for i in range(len(X)):
            y_previsto = prever_perceptron(X[i], w, b)
            erro = y[i] - y_previsto

            
            w = w + eta * erro * X[i]
            b = b + eta * erro

            if erro != 0:
                erros += 1

        print(f"Época {epoca + 1:2d} - erros: {erros}")

       
        if erros == 0:
            break

    return w, b

### Treinando

Usei taxa de aprendizado 0.1 e no máximo 50 épocas.

In [ ]:
w, b = treinar_perceptron(X_treino_fraude, y_treino_fraude, eta=0.1, epocas=50)

print()
print("Pesos finais:", w)
print("Viés final:", b)

Conferindo se ele acerta os próprios dados de treino:

In [ ]:
previsoes = np.array([prever_perceptron(x, w, b) for x in X_treino_fraude])
print("Previsto:", previsoes)
print("Real:    ", y_treino_fraude)

### Testando as transações novas

In [ ]:
transacoes = {
    "Transação A": np.array([2.5, 2.0]),
    "Transação B": np.array([8.0, 6.5])
}

for nome, x in transacoes.items():
    classe = prever_perceptron(x, w, b)
    if classe == 1:
        texto = "Suspeita"
    else:
        texto = "Legítima"
    print(f"{nome} {x} -> classe {classe} ({texto})")

A transação A tem valor e frequência baixos, parecida com as legítimas do treino, e a B está no meio das suspeitas.


## Desafio 2 - KNN (risco de churn)

Características de cada cliente:

- x1: dias de inatividade
- x2: chamados críticos abertos

Classes: 0 (baixo risco) e 1 (alto risco).

O KNN não tem treino, ele guarda os dados e na hora de classificar compara o ponto novo com todos.

In [ ]:
X_treino_churn = np.array([
    [2.0, 1.0], [3.0, 0.0], [5.0, 1.0], [6.0, 2.0],
    [15.0, 4.0], [18.0, 5.0], [20.0, 4.0], [22.0, 6.0]
])
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

### Função de distância

- Euclidiana: raiz da soma dos quadrados das diferenças
- Manhattan: soma dos valores absolutos das diferenças

`X - ponto` funciona por broadcasting, o NumPy subtrai o ponto de todas as linhas de uma vez, então não precisa de `for`.

In [ ]:
def calcular_distancias(X, ponto, metrica="euclidiana"):
    diferenca = X - ponto

    if metrica == "euclidiana":
        return np.sqrt(np.sum(diferenca ** 2, axis=1))
    elif metrica == "manhattan":
        return np.sum(np.abs(diferenca), axis=1)
    else:
        raise ValueError("Métrica inválida, use 'euclidiana' ou 'manhattan'")

### Função KNN



In [ ]:
def knn(X, y, ponto, k, metrica="euclidiana"):
    distancias = calcular_distancias(X, ponto, metrica)

    indices_vizinhos = np.argsort(distancias)[:k]
    classes_vizinhos = y[indices_vizinhos]

    votos = np.bincount(classes_vizinhos)
    classe = np.argmax(votos)

    return classe, indices_vizinhos, distancias[indices_vizinhos]

### Classificando os clientes (K = 3)

In [ ]:
clientes = {
    "Cliente 1": np.array([4.0, 1.0]),
    "Cliente 2": np.array([17.0, 5.0])
}
rotulos = {0: "Baixo Risco", 1: "Alto Risco"}

for metrica in ["euclidiana", "manhattan"]:
    print("Métrica:", metrica)
    for nome, ponto in clientes.items():
        classe, indices, dists = knn(X_treino_churn, y_treino_churn, ponto, k=3, metrica=metrica)
        print(f"  {nome} {ponto}")
        print(f"    Classe predita: {rotulos[classe]}")
        print(f"    Índices dos vizinhos: {indices}")
        print(f"    Distâncias: {np.round(dists, 3)}")
    print()

## Desafio 3 - Recomendador de servidores




In [ ]:
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],      # Micro Instância Web
    [4.0, 8.0, 100.0],     # Standard App Server
    [8.0, 16.0, 250.0],    # Medium Backend & Cache
    [16.0, 32.0, 500.0],   # Database Enterprise
    [32.0, 64.0, 1000.0],  # High Performance Computing
    [64.0, 128.0, 2000.0]  # Big Data & AI Training
])

nomes_servidores = [
    "Micro Instância Web", "Standard App Server",
    "Medium Backend & Cache", "Database Enterprise",
    "High Performance Computing", "Big Data & AI Training"
]

### Função de recomendação




In [ ]:
def recomendar(catalogo, perfil, k, metrica="euclidiana"):
    distancias = calcular_distancias(catalogo, perfil, metrica)
    indices = np.argsort(distancias)[:k]
    return indices, distancias[indices]

### Recomendação para o perfil pedido (K = 2)

In [ ]:
perfil = np.array([12.0, 28.0, 450.0])

indices, dists = recomendar(catalogo_servidores, perfil, k=2)

print(f"Perfil demandado: {perfil[0]:.0f} vCPUs, {perfil[1]:.0f} GB RAM, {perfil[2]:.0f} GB SSD")
print()
print("Ranking de recomendações")
print("-" * 40)

for posicao in range(len(indices)):
    i = indices[posicao]
    vcpus, ram, ssd = catalogo_servidores[i]
    print(f"{posicao + 1}º lugar: {nomes_servidores[i]}")
    print(f"   Especificações: {vcpus:.0f} vCPUs | {ram:.0f} GB RAM | {ssd:.0f} GB SSD")
    print(f"   Distância: {dists[posicao]:.2f}")
    print()